In [1]:
1+1

2

In [2]:
import pandas as pd
from pathlib import Path

csv_path = Path.cwd().parent / "data" / "ab_data.csv"
df = pd.read_csv(csv_path)

print("Dataset loaded successfully!")

Dataset loaded successfully!


In [3]:
# Step 02.07A — Dataset Dimensions

print("========== DATASET DIMENSIONS ==========")

print(f"Total rows: {df.shape[0]:,}")
print(f"Total columns: {df.shape[1]}")

========== DATASET DIMENSIONS ==========
Total rows: 294,478
Total columns: 5


In [4]:
# Step 02.07B — Column Names and Data Types

print("========== COLUMN INFORMATION ==========")

df.info()

========== COLUMN INFORMATION ==========
<class 'pandas.DataFrame'>
RangeIndex: 294478 entries, 0 to 294477
Data columns (total 5 columns):
 #   Column        Non-Null Count   Dtype
---  ------        --------------   -----
 0   user_id       294478 non-null  int64
 1   timestamp     294478 non-null  str  
 2   group         294478 non-null  str  
 3   landing_page  294478 non-null  str  
 4   converted     294478 non-null  int64
dtypes: int64(2), str(3)
memory usage: 11.2 MB


In [5]:
# Step 02.07C — Preview First 10 Records

print("========== FIRST 10 RECORDS ==========")

display(df.head(10))

========== FIRST 10 RECORDS ==========


,user_id,timestamp,group,landing_page,converted
0,851104,2017-01-21 22:11:48.556739,control,old_page,0
1,804228,2017-01-12 08:01:45.159739,control,old_page,0
2,661590,2017-01-11 16:55:06.154213,treatment,new_page,0
3,853541,2017-01-08 18:28:03.143765,treatment,new_page,0
4,864975,2017-01-21 01:52:26.210827,control,old_page,1
5,936923,2017-01-10 15:20:49.083499,control,old_page,0
6,679687,2017-01-19 03:26:46.940749,treatment,new_page,1
7,719014,2017-01-17 01:48:29.539573,control,old_page,0
8,817355,2017-01-04 17:58:08.979471,treatment,new_page,1
9,839785,2017-01-15 18:11:06.610965,treatment,new_page,1


In [6]:
# Step 02.07D — Inspect Unique Category Values

print("========== UNIQUE CATEGORY VALUES ==========")

print("\nExperiment groups:")
print(df["group"].unique())

print("\nLanding pages:")
print(df["landing_page"].unique())

print("\nConversion indicators:")
print(df["converted"].unique())

========== UNIQUE CATEGORY VALUES ==========

Experiment groups:
<StringArray>
['control', 'treatment']
Length: 2, dtype: str

Landing pages:
<StringArray>
['old_page', 'new_page']
Length: 2, dtype: str

Conversion indicators:
[0 1]


In [7]:
# Step 02.07E — Timestamp Format Validation

print("========== TIMESTAMP VALIDATION ==========")

parsed_timestamps = pd.to_datetime(
    df["timestamp"],
    errors="coerce"
)

print(f"Total timestamps: {len(df):,}")
print(f"Valid timestamps: {parsed_timestamps.notna().sum():,}")
print(f"Invalid timestamps: {parsed_timestamps.isna().sum():,}")

print("\nTimestamp data type:")
print(parsed_timestamps.dtype)

========== TIMESTAMP VALIDATION ==========
Total timestamps: 294,478
Valid timestamps: 294,478
Invalid timestamps: 0

Timestamp data type:
datetime64[us]


In [8]:
# Step 02.07F — Dataset Date Range

print("========== DATASET DATE RANGE ==========")

print("Earliest timestamp:", parsed_timestamps.min())
print("Latest timestamp:", parsed_timestamps.max())

print(
    "Time span:",
    parsed_timestamps.max() - parsed_timestamps.min()
)

========== DATASET DATE RANGE ==========
Earliest timestamp: 2017-01-02 13:42:05.378582
Latest timestamp: 2017-01-24 13:41:54.460509
Time span: 21 days 23:59:49.081927


In [9]:
# Step 02.07G — Duplicate User ID Inspection

print("========== USER ID UNIQUENESS ==========")

total_rows = len(df)
unique_users = df["user_id"].nunique()
duplicate_user_rows = df["user_id"].duplicated().sum()

print(f"Total rows: {total_rows:,}")
print(f"Unique user IDs: {unique_users:,}")
print(f"Repeated user ID rows: {duplicate_user_rows:,}")

========== USER ID UNIQUENESS ==========
Total rows: 294,478
Unique user IDs: 290,584
Repeated user ID rows: 3,894


In [10]:
# Step 02.07H — Cross-Group Assignment Check

print("========== CROSS-GROUP ASSIGNMENT ==========")

groups_per_user = df.groupby("user_id")["group"].nunique()

users_in_both_groups = (groups_per_user > 1).sum()

print(f"Total unique users: {len(groups_per_user):,}")
print(f"Users in both groups: {users_in_both_groups:,}")
print(
    f"Users in one group only: "
    f"{(groups_per_user == 1).sum():,}"
)

========== CROSS-GROUP ASSIGNMENT ==========
Total unique users: 290,584
Users in both groups: 1,895
Users in one group only: 288,689


In [11]:
# Step 02.07I — Group and Landing Page Consistency

print("========== GROUP-PAGE CONSISTENCY ==========")

group_page_counts = pd.crosstab(
    df["group"],
    df["landing_page"]
)

display(group_page_counts)

mismatch_mask = (
    ((df["group"] == "control") &
     (df["landing_page"] != "old_page"))
    |
    ((df["group"] == "treatment") &
     (df["landing_page"] != "new_page"))
)

print(
    f"\nMismatched assignment rows: "
    f"{mismatch_mask.sum():,}"
)

========== GROUP-PAGE CONSISTENCY ==========


landing_page,new_page,old_page
group,,
control,1928,145274
treatment,145311,1965



Mismatched assignment rows: 3,893


In [12]:
# Step 02.07J — Exact Duplicate Records

print("========== EXACT DUPLICATE RECORDS ==========")

exact_duplicates = df.duplicated().sum()

print(f"Total rows: {len(df):,}")
print(f"Exact duplicate rows: {exact_duplicates:,}")

print(
    f"Exact duplicate percentage: "
    f"{exact_duplicates / len(df) * 100:.4f}%"
)

========== EXACT DUPLICATE RECORDS ==========
Total rows: 294,478
Exact duplicate rows: 0
Exact duplicate percentage: 0.0000%


In [13]:
# Step 02.07K — Missing Value Audit

print("========== MISSING VALUE AUDIT ==========")

missing_counts = df.isna().sum()

missing_percentages = (
    df.isna().mean() * 100
)

missing_summary = pd.DataFrame({
    "Missing Count": missing_counts,
    "Missing Percentage": missing_percentages
})

display(missing_summary)

print(
    f"\nTotal missing cells: "
    f"{missing_counts.sum():,}"
)

========== MISSING VALUE AUDIT ==========


,Missing Count,Missing Percentage
user_id,0,0.0
timestamp,0,0.0
group,0,0.0
landing_page,0,0.0
converted,0,0.0



Total missing cells: 0


In [14]:
# Step 02.07L — Experiment Group Distribution

print("========== EXPERIMENT GROUP DISTRIBUTION ==========")

group_counts = df["group"].value_counts().sort_index()

group_percentages = (
    df["group"].value_counts(normalize=True)
    .sort_index() * 100
)

group_summary = pd.DataFrame({
    "Record Count": group_counts,
    "Percentage": group_percentages.round(4)
})

display(group_summary)

print(f"\nTotal records: {group_counts.sum():,}")

========== EXPERIMENT GROUP DISTRIBUTION ==========


,Record Count,Percentage
group,,
control,147202,49.9874
treatment,147276,50.0126



Total records: 294,478


In [15]:
# Step 02.07M — Unique Users by Experiment Group

print("========== UNIQUE USERS BY GROUP ==========")

unique_users_by_group = (
    df.groupby("group")["user_id"]
    .nunique()
    .sort_index()
)

display(
    unique_users_by_group.to_frame(
        name="Unique User IDs"
    )
)

print(
    "\nSum of group-specific unique users:",
    f"{unique_users_by_group.sum():,}"
)

print(
    "Overall unique users:",
    f"{df['user_id'].nunique():,}"
)

========== UNIQUE USERS BY GROUP ==========


,Unique User IDs
group,
control,146195
treatment,146284



Sum of group-specific unique users: 292,479
Overall unique users: 290,584


In [16]:
# Step 02.07N — Repeated Users Within Each Group

print("========== WITHIN-GROUP REPEATED USERS ==========")

group_user_counts = (
    df.groupby(["group", "user_id"])
    .size()
)

repeated_group_users = (
    group_user_counts[group_user_counts > 1]
)

print(
    "Group-user combinations with multiple rows:",
    f"{len(repeated_group_users):,}"
)

print(
    "Additional rows beyond first observation:",
    f"{(group_user_counts - 1).clip(lower=0).sum():,}"
)

print(
    "Maximum observations for one group-user combination:",
    group_user_counts.max()
)

========== WITHIN-GROUP REPEATED USERS ==========
Group-user combinations with multiple rows: 1,999
Additional rows beyond first observation: 1,999
Maximum observations for one group-user combination: 2


In [17]:
# Step 02.07O — User Observation Frequency

print("========== USER OBSERVATION FREQUENCY ==========")

observations_per_user = (
    df.groupby("user_id")
    .size()
)

frequency_summary = (
    observations_per_user
    .value_counts()
    .sort_index()
    .rename_axis("Observations per User")
    .to_frame("Number of Users")
)

display(frequency_summary)

print(
    "\nMaximum observations for a single user:",
    observations_per_user.max()
)

print(
    "Users with multiple observations:",
    f"{(observations_per_user > 1).sum():,}"
)

========== USER OBSERVATION FREQUENCY ==========


,Number of Users
Observations per User,
1,286690
2,3894



Maximum observations for a single user: 2
Users with multiple observations: 3,894


In [18]:
# Step 02.07P — Duplicate User Categories

print("========== DUPLICATE USER CATEGORIES ==========")

user_group_counts = (
    df.groupby("user_id")["group"]
    .nunique()
)

cross_group_users = (
    (observations_per_user == 2)
    & (user_group_counts == 2)
).sum()

within_group_users = (
    (observations_per_user == 2)
    & (user_group_counts == 1)
).sum()

print(f"Users repeated across both groups: {cross_group_users:,}")
print(f"Users repeated within one group: {within_group_users:,}")

print(
    f"Total repeated users: "
    f"{cross_group_users + within_group_users:,}"
)

print(
    "Matches previous repeated-user count:",
    cross_group_users + within_group_users
    == (observations_per_user > 1).sum()
)

========== DUPLICATE USER CATEGORIES ==========
Users repeated across both groups: 1,895
Users repeated within one group: 1,999
Total repeated users: 3,894
Matches previous repeated-user count: True


In [19]:
# Step 02.07Q — Repeated User Timestamp Audit

print("========== REPEATED USER TIMESTAMPS ==========")

repeated_rows = df[
    df["user_id"].duplicated(keep=False)
].copy()

timestamp_counts = (
    repeated_rows.groupby("user_id")["timestamp"]
    .nunique()
)

print(f"Repeated users: {len(timestamp_counts):,}")

print(
    "Repeated users with identical timestamps:",
    f"{(timestamp_counts == 1).sum():,}"
)

print(
    "Repeated users with different timestamps:",
    f"{(timestamp_counts == 2).sum():,}"
)

print(
    "Timestamp categories reconcile:",
    len(timestamp_counts)
    == (timestamp_counts == 1).sum()
    + (timestamp_counts == 2).sum()
)

========== REPEATED USER TIMESTAMPS ==========
Repeated users: 3,894
Repeated users with identical timestamps: 0
Repeated users with different timestamps: 3,894
Timestamp categories reconcile: True


In [20]:
# Step 02.07R — Repeated User Outcome Consistency

print("========== REPEATED USER OUTCOME AUDIT ==========")

outcomes_per_user = (
    repeated_rows.groupby("user_id")["converted"]
    .nunique()
)

same_outcome = (outcomes_per_user == 1).sum()
different_outcome = (outcomes_per_user == 2).sum()

print(f"Total repeated users: {len(outcomes_per_user):,}")

print(f"Users with identical outcomes: {same_outcome:,}")

print(f"Users with different outcomes: {different_outcome:,}")

print(
    "Outcome categories reconcile:",
    same_outcome + different_outcome
    == len(outcomes_per_user)
)

========== REPEATED USER OUTCOME AUDIT ==========
Total repeated users: 3,894
Users with identical outcomes: 3,030
Users with different outcomes: 864
Outcome categories reconcile: True


In [21]:
# Step 02.07S — Chronological Repeated-User Patterns

print("========== CHRONOLOGICAL REPEATED-USER AUDIT ==========")

chronological_rows = repeated_rows.copy()

chronological_rows["parsed_timestamp"] = pd.to_datetime(
    chronological_rows["timestamp"],
    errors="coerce"
)

chronological_rows = chronological_rows.sort_values(
    ["user_id", "parsed_timestamp"]
)

first_observations = (
    chronological_rows.groupby("user_id").first()
)

second_observations = (
    chronological_rows.groupby("user_id").last()
)

print(f"Repeated users: {len(first_observations):,}")

print(
    "Users changing experiment groups:",
    (
        first_observations["group"]
        != second_observations["group"]
    ).sum()
)

print(
    "Users changing landing pages:",
    (
        first_observations["landing_page"]
        != second_observations["landing_page"]
    ).sum()
)

print(
    "Users changing conversion outcomes:",
    (
        first_observations["converted"]
        != second_observations["converted"]
    ).sum()
)

print(
    "Chronological records reconcile:",
    len(first_observations)
    == len(second_observations)
    == 3894
)

========== CHRONOLOGICAL REPEATED-USER AUDIT ==========
Repeated users: 3,894
Users changing experiment groups: 1895
Users changing landing pages: 1998
Users changing conversion outcomes: 864
Chronological records reconcile: True


In [22]:
# Step 02.07T — User-Level Assignment Categories

print("========== USER-LEVEL ASSIGNMENT AUDIT ==========")

user_group_sets = (
    df.groupby("user_id")["group"]
    .agg(lambda values: frozenset(values))
)

control_only = (
    user_group_sets == frozenset(["control"])
).sum()

treatment_only = (
    user_group_sets == frozenset(["treatment"])
).sum()

both_groups = (
    user_group_sets == frozenset(["control", "treatment"])
).sum()

print(f"Control-only users: {control_only:,}")
print(f"Treatment-only users: {treatment_only:,}")
print(f"Cross-group users: {both_groups:,}")

print(f"Total unique users: {len(user_group_sets):,}")

print(
    "Assignment categories reconcile:",
    control_only + treatment_only + both_groups
    == len(user_group_sets)
)

========== USER-LEVEL ASSIGNMENT AUDIT ==========
Control-only users: 144,300
Treatment-only users: 144,389
Cross-group users: 1,895
Total unique users: 290,584
Assignment categories reconcile: True


In [23]:
# Step 02.07U — User-Level Group-Page Mismatch Audit

print("========== USER-LEVEL GROUP-PAGE MISMATCH AUDIT ==========")

mismatch_mask = (
    ((df["group"] == "control") &
     (df["landing_page"] != "old_page"))
    |
    ((df["group"] == "treatment") &
     (df["landing_page"] != "new_page"))
)

mismatched_rows = df.loc[mismatch_mask]

affected_users = mismatched_rows["user_id"].nunique()

print(f"Mismatched rows: {len(mismatched_rows):,}")

print(f"Unique users affected: {affected_users:,}")

print(
    "Affected users with cross-group assignments:",
    mismatched_rows.loc[
        mismatched_rows["user_id"].isin(
            user_group_sets[
                user_group_sets.apply(lambda x: len(x) > 1)
            ].index
        ),
        "user_id"
    ].nunique()
)

print(
    "Mismatch row count reconciles:",
    len(mismatched_rows) == 3893
)

========== USER-LEVEL GROUP-PAGE MISMATCH AUDIT ==========
Mismatched rows: 3,893
Unique users affected: 3,893
Affected users with cross-group assignments: 1895
Mismatch row count reconciles: True


In [24]:
# Step 02.07V — Data Quality Issue Overlap

print("========== DATA QUALITY ISSUE OVERLAP ==========")

mismatch_users = set(mismatched_rows["user_id"])

cross_group_users = set(
    user_group_sets[
        user_group_sets.apply(lambda groups: len(groups) > 1)
    ].index
)

within_group_repeated_users = set(
    observations_per_user[
        observations_per_user > 1
    ].index
) - cross_group_users

all_affected_users = (
    mismatch_users
    | cross_group_users
    | within_group_repeated_users
)

print(f"Mismatch-affected users: {len(mismatch_users):,}")

print(f"Cross-group users: {len(cross_group_users):,}")

print(
    "Within-group repeated users:",
    f"{len(within_group_repeated_users):,}"
)

print(
    "Total distinct users with any identified issue:",
    f"{len(all_affected_users):,}"
)

print(
    "Cross-group users included among mismatch users:",
    cross_group_users.issubset(mismatch_users)
)

print(
    "Affected-user count does not exceed total users:",
    len(all_affected_users) <= df["user_id"].nunique()
)

========== DATA QUALITY ISSUE OVERLAP ==========
Mismatch-affected users: 3,893
Cross-group users: 1,895
Within-group repeated users: 1,999
Total distinct users with any identified issue: 3,894
Cross-group users included among mismatch users: True
Affected-user count does not exceed total users: True


In [25]:
# Step 02.07W — Unaffected User Population

print("========== UNAFFECTED USER POPULATION ==========")

all_users = set(df["user_id"])

unaffected_users = all_users - all_affected_users

print(f"Total unique users: {len(all_users):,}")

print(
    "Users with identified issues:",
    f"{len(all_affected_users):,}"
)

print(
    "Users without identified issues:",
    f"{len(unaffected_users):,}"
)

print(
    "Population reconciles:",
    len(unaffected_users) + len(all_affected_users)
    == len(all_users)
)

print(
    "Affected and unaffected groups are disjoint:",
    all_affected_users.isdisjoint(unaffected_users)
)

========== UNAFFECTED USER POPULATION ==========
Total unique users: 290,584
Users with identified issues: 3,894
Users without identified issues: 286,690
Population reconciles: True
Affected and unaffected groups are disjoint: True


In [26]:
# Step 02.07X — Unaffected Population by Group

print("========== UNAFFECTED USERS BY GROUP ==========")

unaffected_df = df[
    df["user_id"].isin(unaffected_users)
].copy()

unaffected_group_counts = (
    unaffected_df.groupby("group")["user_id"]
    .nunique()
    .sort_index()
)

display(
    unaffected_group_counts.to_frame(
        name="Unique Users"
    )
)

print(
    "Total unaffected users:",
    f"{unaffected_group_counts.sum():,}"
)

print(
    "Group totals reconcile:",
    unaffected_group_counts.sum()
    == len(unaffected_users)
)

print(
    "One observation per unaffected user:",
    len(unaffected_df)
    == len(unaffected_users)
)

========== UNAFFECTED USERS BY GROUP ==========


,Unique Users
group,
control,143293
treatment,143397


Total unaffected users: 286,690
Group totals reconcile: True
One observation per unaffected user: True


In [27]:
# Step 02.07Y — Original Dataset Integrity Check

print("========== ORIGINAL DATASET INTEGRITY ==========")

expected_columns = [
    "user_id",
    "timestamp",
    "group",
    "landing_page",
    "converted"
]

print(f"Current rows: {len(df):,}")
print(f"Current columns: {len(df.columns)}")

print(
    "Original row count preserved:",
    len(df) == 294478
)

print(
    "Original column structure preserved:",
    df.columns.tolist() == expected_columns
)

print(
    "Original dataset has no missing values:",
    df.isna().sum().sum() == 0
)

print(
    "Original dataset has no exact duplicates:",
    df.duplicated().sum() == 0
)

========== ORIGINAL DATASET INTEGRITY ==========
Current rows: 294,478
Current columns: 5
Original row count preserved: True
Original column structure preserved: True
Original dataset has no missing values: True
Original dataset has no exact duplicates: True
